# Image Description with the **Responses API** (Azure OpenAI)

This notebook demonstrates how to use the **vision capabilities** of the Responses API to analyse and describe images.
We pass an image to the model — either via a **public URL** or as a **base64-encoded string** — and ask it to describe what it sees.

---

## What you will learn

- Load environment variables and initialise the `AzureOpenAI` client
- Send a `responses.create()` call with **image input** (URL)
- Send a `responses.create()` call with **image input** (local file encoded as base64)
- Read and display the model's textual description

---

## Prerequisites

- Python 3.10+
- `openai` package installed (`pip install openai`)
- Azure OpenAI resource with a **vision-capable** model deployment (e.g., `gpt-4o`, `gpt-4.1`)
- Environment variables set:
  - `AZURE_OPENAI_API_KEY`
  - `AZURE_OPENAI_ENDPOINT` (e.g., `https://<your-resource>.openai.azure.com`)
  - `AZURE_OPENAI_API_VERSION` (e.g., `2025-04-01-preview`)
  - `MODEL_DEPLOYMENT_NAME` (the *deployment name* of your vision model)

## [Disclaimer by OpenAI](https://platform.openai.com/docs/assistants/migration?user-chat-app=responses)
*After achieving feature parity in the Responses API, we've deprecated the Assistants API. It will shut down on August 26, 2026. Follow the guidance below to update your integration. [Learn more](https://platform.openai.com/docs/guides/migrate-to-responses).*

# Constants and Libraries

In [ ]:
import os
import base64
from IPython.display import Markdown, Image, display
from dotenv import load_dotenv  # package python-dotenv
from openai import AzureOpenAI  # package openai

if not load_dotenv("./../../config/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

messages_system = "You are an AI assistant that analyses and describes images in detail."

# Public image URL to describe
image_url = "https://github.com/user-attachments/assets/4747c8ac-ea32-459c-83fb-7a30571ad214"

# Create a client to connect to Azure OpenAI service and deployment

In [ ]:
client = AzureOpenAI(
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY")
)

print(f"OpenAI Endpoint base_url: {client.base_url}")

# Describe an Image from a URL

Pass the image as an `image_url` content block inside the user message.
The model will return a detailed textual description of what it sees.

In [ ]:
response = client.responses.create(
    model=os.environ["MODEL_DEPLOYMENT_NAME"],
    input=[
        {
            "role": "system",
            "content": messages_system
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "Please describe this image in detail."
                },
                {
                    "type": "input_image",
                    "image_url": image_url
                }
            ]
        }
    ]
)

description = response.output[0].content[0].text
print("Response ID:", response.id)
print()
display(Markdown(description))

# Describe an Image from a Local File (base64)

For images that are **not publicly accessible**, encode them as base64 and embed the data directly in the request.

In [ ]:
# Replace with the path to your local image file
local_image_path = "code_interpreter_image_1.png"

with open(local_image_path, "rb") as image_file:
    image_data = base64.b64encode(image_file.read()).decode("utf-8")

# Detect MIME type from file extension
ext = os.path.splitext(local_image_path)[-1].lower()
mime_types = {".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".gif": "image/gif", ".webp": "image/webp"}
mime_type = mime_types.get(ext, "image/png")

response_local = client.responses.create(
    model=os.environ["MODEL_DEPLOYMENT_NAME"],
    input=[
        {
            "role": "system",
            "content": messages_system
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "Please describe this image in detail."
                },
                {
                    "type": "input_image",
                    "image_url": f"data:{mime_type};base64,{image_data}"
                }
            ]
        }
    ]
)

description_local = response_local.output[0].content[0].text
print("Response ID:", response_local.id)
print()
display(Image(filename=local_image_path))
print()
display(Markdown(description_local))